# AJ's dual T4 entity-resolution experiment

Attach the official `student_resource` archive as a **private Kaggle Dataset**, select **GPU T4 x2**, and run the notebook. This single code cell discovers the attached `dataset/` directory, generates bounded candidates for all test Source 1 records, fits separate Source 2 and Source 3 XGBoost models concurrently on `cuda:0` and `cuda:1`, tunes macro F0.5 on held-out training Source 1 records, and writes both required TSVs under `/kaggle/working/aj_entity_resolution/output/`.

Candidate generation and output assembly use CPU/DuckDB. Output assembly sorts only IDs, with one thread and a 768 MB DuckDB budget, then streams one Source 1 row at a time. This budget is not a hard cap on total process memory. Completed inference partitions receive a completion record. The final output record is `output_completion.json`; `run_manifest.json` also records metrics and package versions.

**If your previous run finished inference and died during export, use `aj_recover_outputs.ipynb` instead. Do not rerun this full cell to recover files.**

Inspect measured candidate recall, validation score, runtime, disk use, and output validation before uploading. Challenge data, predictions, and models stay private. The recovery change has not yet completed a full Kaggle run; no leaderboard score is claimed. Both TSVs receive streaming checks, including target ID existence. The separate organizer validator is not run automatically because it materializes large Python collections.


In [ ]:
"""Memory-bounded submission assembly and recovery from completed Parquet files.

This module is embedded into both notebooks by sync_notebook.py. It imports no
GPU framework and never trains a model or changes the accepted candidate set.
"""

import gc
import hashlib
import heapq
import itertools
import json
import os
import shutil
import time
from pathlib import Path

import duckdb
import pyarrow.parquet as pq


ASSEMBLY_VERSION = "streaming-v1"


def ao_log(message):
    print(time.strftime("%Y-%m-%d %H:%M:%S"), message, flush=True)


def ao_quote(value):
    return "'" + str(value).replace("'", "''") + "'"


def ao_file_list(paths):
    return "[" + ",".join(ao_quote(p) for p in paths) + "]"


def ao_csv(path):
    return f"read_csv({ao_quote(path)}, delim='\\t', header=true, all_varchar=true)"


def ao_resources(work):
    free_gb = shutil.disk_usage(work).free / 1024**3
    rss = "unavailable"
    try:
        import psutil
        rss = f"{psutil.Process().memory_info().rss / 1024**3:.2f} GiB"
    except ImportError:
        pass
    ao_log(f"Resources: process RAM={rss}; disk free={free_gb:.2f} GiB")


def ao_connection(work, memory_mb=768):
    # Only external sorts and ordinary joins/counts run here. No list/string
    # aggregate, dataframe materialization, or Python-wide ID set is used.
    tmp = work / "assembly_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(work).free
    if free < 512 * 1024**2:
        raise RuntimeError("Less than 512 MiB disk free. Preserve the original Parquets and free scratch space before recovery.")
    db = duckdb.connect()
    try:
        db.execute(f"SET memory_limit = '{int(memory_mb)}MB'")
        db.execute("SET threads = 1")
        db.execute("SET preserve_insertion_order = true")
        db.execute(f"SET temp_directory = {ao_quote(tmp)}")
        spill_bytes = min(4 * 1024**3, max(256 * 1024**2, free // 2))
        db.execute(f"SET max_temp_directory_size = '{spill_bytes}B'")
        return db
    except Exception:
        db.close()
        raise


def ao_atomic_json(path, value):
    partial = path.with_name(path.name + ".partial")
    partial.write_text(json.dumps(value, indent=2), encoding="utf-8")
    os.replace(partial, path)


def ao_fingerprint(path):
    path = Path(path).resolve()
    stat = path.stat()
    return {"path": str(path), "bytes": stat.st_size, "mtime_ns": stat.st_mtime_ns}


def ao_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def ao_parquet_info(path, required):
    if not Path(path).is_file():
        raise RuntimeError(f"Required saved file is missing: {path}. Recovery cannot recreate missing predictions.")
    try:
        with pq.ParquetFile(path) as reader:
            missing = set(required) - set(reader.schema_arrow.names)
            if missing:
                raise ValueError(f"Missing columns: {sorted(missing)}")
            return reader.metadata.num_rows
    except Exception as error:
        raise RuntimeError(f"Saved Parquet is incomplete or invalid: {path}. Preserve the other files; this partition needs regeneration.") from error


def ao_sorted_file(path, work, pair_file=True):
    path = Path(path)
    identity = {"assembly_version": ASSEMBLY_VERSION, "pair_file": pair_file,
                "input": ao_fingerprint(path)}
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]
    cache = work / "assembly_sorted"
    cache.mkdir(exist_ok=True)
    out = cache / f"{digest}.parquet"
    marker = cache / f"{digest}.json"
    columns = ["qid", "tid"] if pair_file else ["qid"]
    expected_rows = ao_parquet_info(path, columns) if pair_file else None
    if out.exists() and marker.exists():
        saved = json.loads(marker.read_text(encoding="utf-8"))
        actual_rows = ao_parquet_info(out, columns)
        if saved.get("identity") == identity and actual_rows == saved.get("rows"):
            if expected_rows is None or actual_rows == expected_rows:
                ao_log(f"Reuse sorted IDs: {path.name} ({actual_rows:,} rows)")
                return out, actual_rows
    ao_resources(work)
    ao_log(f"External sort with one CPU thread and 768 MB DuckDB budget: {path.name}")
    partial = out.with_name(out.stem + ".partial.parquet")
    db = ao_connection(work)
    try:
        if pair_file:
            query = f"SELECT qid, tid FROM read_parquet({ao_quote(path)}) ORDER BY qid, tid"
        else:
            expected_rows = db.execute(f"SELECT count(*) FROM {ao_csv(path)}").fetchone()[0]
            query = f"SELECT entity_id AS qid FROM {ao_csv(path)} ORDER BY entity_id"
        db.execute(f"COPY ({query}) TO {ao_quote(partial)} (FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 65536)")
    finally:
        db.close()
        gc.collect()
    actual_rows = ao_parquet_info(partial, columns)
    if actual_rows != expected_rows:
        raise RuntimeError(f"Sorted row count changed for {path}: {expected_rows} -> {actual_rows}")
    os.replace(partial, out)
    ao_atomic_json(marker, {"identity": identity, "rows": actual_rows})
    return out, actual_rows


def ao_rows(path, columns):
    previous = None
    with pq.ParquetFile(path) as reader:
        for batch in reader.iter_batches(batch_size=8192, columns=columns, use_threads=False):
            values = [batch.column(i).to_pylist() for i in range(len(columns))]
            for row in zip(*values):
                if any(value is None or not isinstance(value, str) for value in row):
                    raise RuntimeError(f"Null or non-string identifier in {path}")
                if previous is not None and row < previous:
                    raise RuntimeError(f"Sorted cache is out of order: {path}")
                previous = row
                yield row


def ao_grouped_pairs(sorted_paths, maximum):
    streams = [ao_rows(path, ["qid", "tid"]) for path in sorted_paths]
    try:
        merged = heapq.merge(*streams)
        for qid, rows in itertools.groupby(merged, key=lambda row: row[0]):
            ids = []
            previous = None
            for _, tid in rows:
                if tid == previous:
                    raise RuntimeError(f"Duplicate pair encountered for {qid}; no pairs were silently removed.")
                if not tid.startswith(("S2-", "S3-")) or any(c in tid for c in ",\t\r\n"):
                    raise RuntimeError(f"Invalid target ID for {qid}: {tid!r}")
                ids.append(tid)
                if len(ids) > maximum:
                    raise RuntimeError(f"{qid} exceeds the configured {maximum}-candidate bound. Check the selected run files.")
                previous = tid
            yield qid, ids
    finally:
        for stream in streams:
            stream.close()


def ao_check_target_ids(candidate_parts, test_dir, work):
    for source, prefix in (("source2", "S2-"), ("source3", "S3-")):
        parts = [p for p in candidate_parts if p.name.startswith(source + "_")]
        if not parts:
            raise RuntimeError(f"No {source} candidate partitions were selected")
        target_path = test_dir / f"test_{source}.tsv"
        if not target_path.is_file():
            raise RuntimeError(f"Missing official target source: {target_path}")
        ao_log(f"Check candidate IDs against the official test {source} file")
        db = ao_connection(work)
        try:
            bad_prefix = db.execute(f"SELECT count(*) FROM read_parquet({ao_file_list(parts)}) WHERE tid IS NULL OR NOT starts_with(tid, {ao_quote(prefix)})").fetchone()[0]
            invalid = db.execute(f"""
              SELECT count(*) FROM read_parquet({ao_file_list(parts)}) p
              ANTI JOIN {ao_csv(target_path)} t ON p.tid = t.entity_id
            """).fetchone()[0]
            if bad_prefix or invalid:
                raise RuntimeError(f"{source}: wrong-prefix rows={bad_prefix}, unknown-target rows={invalid}")
        finally:
            db.close()
            gc.collect()


def assemble_outputs(candidate_parts, accepted_parts, data_root, work_root,
                     max_candidates_per_entity=80, expected_accepted_counts=None):
    """Sort narrow pair files sequentially; merge/write one Source 1 at a time."""
    data_root, work = Path(data_root), Path(work_root)
    candidate_parts = [Path(p) for p in candidate_parts]
    accepted_parts = [Path(p) for p in accepted_parts]
    if not candidate_parts or len(candidate_parts) != len(accepted_parts):
        raise RuntimeError("Need a complete candidate/accepted partition pair for every source and country")
    if len({p.name for p in candidate_parts}) != len(candidate_parts):
        raise RuntimeError("Duplicate candidate partition names")
    if sorted(p.name for p in candidate_parts) != sorted(p.name for p in accepted_parts):
        raise RuntimeError("Candidate and accepted partition filenames do not describe the same run")
    candidate_count = sum(ao_parquet_info(p, ["qid", "tid"]) for p in candidate_parts)
    accepted_count = sum(ao_parquet_info(p, ["qid", "tid"]) for p in accepted_parts)
    expected_accepted_counts = expected_accepted_counts or {}
    candidate_by_name = {p.name: p for p in candidate_parts}
    completion_evidence = {}
    sidecar_flags = [p.with_name(p.name + ".complete.json").exists() for p in accepted_parts]
    if any(sidecar_flags) and not all(sidecar_flags):
        raise RuntimeError("Mixed old and new inference checkpoints. Keep separate runs together; do not assemble a mixture.")
    saved_thresholds = set()
    source_models = {}
    for part in accepted_parts:
        count = ao_parquet_info(part, ["qid", "tid"])
        sidecar = part.with_name(part.name + ".complete.json")
        candidate = candidate_by_name[part.name]
        if sidecar.exists():
            evidence = json.loads(sidecar.read_text(encoding="utf-8"))
            expected_rows = ao_parquet_info(candidate, ["qid", "tid"])
            # Content identity survives restoring the run under a different path.
            if (evidence.get("candidate_sha256") != ao_sha256(candidate)
                    or evidence.get("accepted_rows") != count
                    or evidence.get("scored_rows") != expected_rows):
                raise RuntimeError(f"Inference completion record does not match {part.name}")
            source = part.name.split("_", 1)[0]
            if evidence.get("model_file") != source + ".json":
                raise RuntimeError(f"Wrong source model in checkpoint: {part.name}")
            model_path = work / "models" / evidence["model_file"]
            if not model_path.is_file() or ao_sha256(model_path) != evidence["model_sha256"]:
                raise RuntimeError(f"Saved model changed after inference for {part.name}")
            if ao_sha256(part) != evidence.get("accepted_sha256"):
                raise RuntimeError(f"Accepted file changed after inference for {part.name}")
            saved_threshold = evidence.get("threshold")
            rounds = evidence.get("prediction_rounds")
            if not isinstance(saved_threshold, (int, float)) or not 0 <= saved_threshold <= 1:
                raise RuntimeError(f"Invalid threshold in checkpoint: {part.name}")
            if not isinstance(rounds, int) or rounds < 1:
                raise RuntimeError(f"Invalid prediction rounds in checkpoint: {part.name}")
            identity = (evidence["model_sha256"], rounds)
            if source_models.setdefault(source, identity) != identity:
                raise RuntimeError(f"Different models or prediction rounds for {source}")
            saved_thresholds.add(saved_threshold)
            completion_evidence[part.name] = evidence
        elif part.name in expected_accepted_counts:
            expected = expected_accepted_counts[part.name]
            if count != expected:
                raise RuntimeError(f"{part.name}: {count:,} rows, but completed inference log reported {expected:,}; do not use this partial file")
            completion_evidence[part.name] = {
                "accepted_rows": count, "evidence": "matched AJ's supplied completed-inference log",
                "model_and_threshold_hashes": "not available for this legacy checkpoint",
            }
        else:
            raise RuntimeError(f"No completion record for {part.name}. Supply its accepted count from a completed inference log; a valid footer alone is insufficient.")
    if len(saved_thresholds) > 1:
        raise RuntimeError("Inference checkpoints use different thresholds; do not combine runs")
    ao_log(f"ASSEMBLY START: {candidate_count:,} candidate pairs; {accepted_count:,} accepted pairs")
    ao_resources(work)
    source_files = [data_root / "test" / f"test_source{i}.tsv" for i in (1, 2, 3)]
    input_files = candidate_parts + accepted_parts + source_files
    snapshots = [ao_fingerprint(p) for p in input_files]
    # An interrupted rerun must not leave a stale success marker for new files.
    (work / "output_completion.json").unlink(missing_ok=True)
    ao_check_target_ids(candidate_parts, data_root / "test", work)
    source1 = data_root / "test" / "test_source1.tsv"
    sorted_s1, expected_s1 = ao_sorted_file(source1, work, pair_file=False)
    sorted_candidates = [ao_sorted_file(p, work)[0] for p in candidate_parts]
    sorted_accepted = [ao_sorted_file(p, work)[0] for p in accepted_parts]

    output = work / "output"
    output.mkdir(exist_ok=True)
    candidate_final = output / "candidate_pairs.tsv"
    matching_final = output / "matching_results.tsv"
    candidate_partial = output / "candidate_pairs.tsv.partial"
    matching_partial = output / "matching_results.tsv.partial"
    chash, mhash = hashlib.sha256(), hashlib.sha256()
    cgroups = ao_grouped_pairs(sorted_candidates, max_candidates_per_entity)
    mgroups = ao_grouped_pairs(sorted_accepted, max_candidates_per_entity)
    source_rows = ao_rows(sorted_s1, ["qid"])
    cgroup, mgroup = next(cgroups, None), next(mgroups, None)
    rows = seen_candidates = seen_matches = empty_candidates = 0
    previous_qid = None
    ao_log("Stream both TSV files; validate coverage, duplicates and match/candidate subsets")
    try:
        with candidate_partial.open("wb", buffering=1024**2) as cf, matching_partial.open("wb", buffering=1024**2) as mf:
            cheader = b"source1_entity_id\tcandidate_entity_ids\n"
            mheader = b"source1_entity_id\tmatched_entity_ids\n"
            cf.write(cheader); chash.update(cheader)
            mf.write(mheader); mhash.update(mheader)
            for (qid,) in source_rows:
                if qid == previous_qid or not qid.startswith("S1-") or any(c in qid for c in ",\t\r\n"):
                    raise RuntimeError("Invalid or duplicated Source 1 ID in official input")
                if cgroup is not None and cgroup[0] < qid:
                    raise RuntimeError(f"Candidate Source 1 ID absent from test: {cgroup[0]}")
                if mgroup is not None and mgroup[0] < qid:
                    raise RuntimeError(f"Accepted Source 1 ID absent from test: {mgroup[0]}")
                cids, mids = [], []
                if cgroup is not None and cgroup[0] == qid:
                    cids = cgroup[1]
                    cgroup = next(cgroups, None)
                if mgroup is not None and mgroup[0] == qid:
                    mids = mgroup[1]
                    mgroup = next(mgroups, None)
                if not set(mids).issubset(cids):
                    raise RuntimeError(f"Accepted ID missing from the scored candidate set for {qid}")
                cline = (qid + "\t" + ",".join(cids) + "\n").encode("utf-8")
                mline = (qid + "\t" + ",".join(mids) + "\n").encode("utf-8")
                cf.write(cline); chash.update(cline)
                mf.write(mline); mhash.update(mline)
                rows += 1
                seen_candidates += len(cids)
                seen_matches += len(mids)
                empty_candidates += not cids
                previous_qid = qid
                if rows % 100_000 == 0:
                    ao_log(f"  assembled {rows:,}/{expected_s1:,} Source 1 entities")
            if cgroup is not None or mgroup is not None:
                raise RuntimeError("Pairs remain for Source 1 IDs absent from the official test source")
            if (rows, seen_candidates, seen_matches) != (expected_s1, candidate_count, accepted_count):
                raise RuntimeError("Row totals differ from the complete input Parquet/source counts")
            cf.flush(); os.fsync(cf.fileno())
            mf.flush(); os.fsync(mf.fileno())
    finally:
        source_rows.close(); cgroups.close(); mgroups.close()
    if snapshots != [ao_fingerprint(p) for p in input_files]:
        raise RuntimeError("Input files changed during assembly; no completion marker was written")
    os.replace(candidate_partial, candidate_final)
    os.replace(matching_partial, matching_final)
    result = {
        "assembly_version": ASSEMBLY_VERSION, "source1_rows": rows,
        "candidate_pairs": seen_candidates, "accepted_pairs": seen_matches,
        "source1_without_candidates": empty_candidates,
        "validation": "PASS: exact Source 1 coverage, target existence, unique pairs and match subsets",
        "official_validator_run": False,
        "inference_completion_evidence": completion_evidence,
        "input_files": snapshots,
        "outputs": {
            candidate_final.name: {"bytes": candidate_final.stat().st_size, "sha256": chash.hexdigest()},
            matching_final.name: {"bytes": matching_final.stat().st_size, "sha256": mhash.hexdigest()},
        },
    }
    ao_atomic_json(work / "output_completion.json", result)
    ao_log(f"OUTPUT COMPLETE: {rows:,} Source 1 rows, {seen_matches:,} final links. Files: {output}")
    ao_resources(work)
    return result


def recover_saved_outputs(data_root=None, work_root=None,
                          candidate_tag="v3_64_512_3_40", expected_accepted_counts=None):
    work = Path(work_root or "/kaggle/working/aj_entity_resolution")
    if not work.is_dir():
        raise RuntimeError(f"Saved run directory is missing: {work}. Restore its Parquets before recovery.")
    if data_root is None:
        roots = sorted({p.parent.parent.resolve() for p in Path("/kaggle/input").glob("**/dataset/test/test_source1.tsv")})
        if len(roots) != 1:
            raise RuntimeError(f"Expected one attached official dataset; found {roots}. Set data_root explicitly.")
        data_root = roots[0]
    data_root = Path(data_root)
    db = ao_connection(work)
    try:
        countries = [row[0] for row in db.execute(f"SELECT DISTINCT country FROM {ao_csv(data_root / 'test' / 'test_source1.tsv')} ORDER BY country").fetchall()]
    finally:
        db.close()
    candidates, accepted = [], []
    for country in countries:
        country_key = hashlib.sha1(country.encode()).hexdigest()[:10]
        for source in ("source2", "source3"):
            filename = f"{source}_{country_key}_{candidate_tag}.parquet"
            candidates.append(work / "test_candidates" / filename)
            accepted.append(work / "accepted" / filename)
    ao_log("RECOVERY ONLY: reuse existing Parquets; no training, inference or GPU imports")
    return assemble_outputs(candidates, accepted, data_root, work,
                            expected_accepted_counts=expected_accepted_counts)


"""AJ's Kaggle experiment, embedded with output_assembly.py in one notebook cell.

Official challenge data stays outside Git. This is an experimental, measurable
pipeline, not a claim of a winning or Kaggle-verified score.
"""

import gc
import hashlib
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path


import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import xgboost as xgb



SEED = 2026
QUERY_SAMPLE_MOD = 15  # 5% validation, 10% training; all target rows remain searchable.
VALID_MOD = 5
MAX_TOKEN_DF = 64
MAX_EXACT_DF = 512
MAX_TOKEN_KEYS_PER_CHANNEL = 3
MAX_CANDIDATES_PER_SOURCE = 40
CANDIDATE_VERSION = "v3"
DUCKDB_MEMORY = "9GB"
THREADS = max(2, min(8, os.cpu_count() or 4))
FEATURES = [
    "name_jw", "address_jw", "name_jaccard", "address_jaccard",
    "name_length_ratio", "address_length_ratio", "number_equal",
    "number_both_present", "exact_name", "exact_address",
    "key_hits", "name_key", "address_key", "number_address_key",
]


def locate_dataset():
    roots = [Path("/kaggle/input"), Path.cwd()]
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.glob("**/dataset/train/train_ground_truth.tsv"))
    roots_found = sorted({p.parent.parent.resolve() for p in candidates})
    if len(roots_found) != 1:
        raise RuntimeError(
            f"Expected one official dataset under Kaggle input or cwd; found {roots_found}. "
            "Set DATA_ROOT explicitly if you attached more than one copy."
        )
    return roots_found[0]


DATA_ROOT = locate_dataset()
WORK = Path("/kaggle/working/aj_entity_resolution") if Path("/kaggle/working").exists() else Path.cwd() / "work" / "aj_entity_resolution"
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "tmp").mkdir(exist_ok=True)
(WORK / "models").mkdir(exist_ok=True)
(WORK / "train_candidates").mkdir(exist_ok=True)
(WORK / "test_candidates").mkdir(exist_ok=True)
(WORK / "accepted").mkdir(exist_ok=True)
OUTPUT = WORK / "output"
OUTPUT.mkdir(exist_ok=True)


def log(message):
    print(time.strftime("%Y-%m-%d %H:%M:%S"), message, flush=True)


if torch.cuda.device_count() < 2:
    raise RuntimeError("This run requires Kaggle's 2 x T4 GPU accelerator. Enable it and restart.")
log(f"Data: {DATA_ROOT}; GPUs: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_name(1)}")
log(f"Versions: DuckDB {duckdb.__version__}, XGBoost {xgb.__version__}, Torch {torch.__version__}")


def sq(value):
    return "'" + str(value).replace("'", "''") + "'"


def con(memory=DUCKDB_MEMORY):
    db = duckdb.connect()
    db.execute(f"SET memory_limit = {sq(memory)}")
    db.execute(f"SET threads = {THREADS}")
    db.execute(f"SET temp_directory = {sq(WORK / 'tmp')}")
    db.execute("SET preserve_insertion_order = false")
    return db


def csv_source(path):
    return f"read_csv({sq(path)}, delim='\\t', header=true, all_varchar=true)"


def normalized_record_sql(path, country, is_query, sampled):
    # Generic accent folding and punctuation treatment; no country whitelist.
    filt = f"country = {sq(country)}"
    if is_query and sampled:
        filt += f" AND hash(entity_id) % 100 < {QUERY_SAMPLE_MOD}"
    src = csv_source(path)
    return f"""
        SELECT entity_id AS rid,
          trim(regexp_replace(regexp_replace(
            ' ' || trim(regexp_replace(lower(strip_accents(coalesce(business_name,''))), '[^a-z0-9]+', ' ', 'g')) || ' ',
            ' (incorporated|inc|corporation|corp|limited|ltd|llc|llp|private|pvt) ', ' ', 'g'),
            ' +', ' ', 'g')) AS nm,
          trim(regexp_replace(lower(strip_accents(coalesce(business_address,''))), '[^a-z0-9]+', ' ', 'g')) AS ad,
          regexp_extract(coalesce(business_address,''), '[0-9]+') AS num
        FROM {src}
        WHERE {filt}
    """


def keys_sql(table):
    # Separate exact, name-token, address-token, and house-number/address-token
    # channels. The document-frequency gate below prevents giant Cartesian blocks.
    return f"""
      SELECT DISTINCT rid, key FROM (
        SELECT rid, 'N=' || nm AS key FROM {table} WHERE length(nm) >= 3
        UNION ALL
        SELECT rid, 'A=' || ad AS key FROM {table} WHERE length(ad) >= 5
        UNION ALL
        SELECT rid, 'NT=' || tok AS key
          FROM {table}, UNNEST(str_split(nm, ' ')) AS u(tok)
          WHERE length(tok) >= 3 AND regexp_matches(tok, '[a-z]')
        UNION ALL
        SELECT rid, 'AT=' || tok AS key
          FROM {table}, UNNEST(str_split(ad, ' ')) AS u(tok)
          WHERE length(tok) >= 4 AND regexp_matches(tok, '[a-z]')
        UNION ALL
        SELECT rid, 'AN=' || num || ':' || tok AS key
          FROM {table}, UNNEST(str_split(ad, ' ')) AS u(tok)
          WHERE length(num) >= 2 AND length(tok) >= 4 AND regexp_matches(tok, '[a-z]')
      )
    """


def build_partition(split, country, source):
    folder = WORK / f"{split}_candidates"
    safe_country = hashlib.sha1(country.encode()).hexdigest()[:10]
    out = folder / f"{source}_{safe_country}_{CANDIDATE_VERSION}_{MAX_TOKEN_DF}_{MAX_EXACT_DF}_{MAX_TOKEN_KEYS_PER_CHANNEL}_{MAX_CANDIDATES_PER_SOURCE}.parquet"
    if out.exists():
        log(f"Reuse candidate partition: {out.name}")
        return out
    qpath = DATA_ROOT / split / f"{split}_source1.tsv"
    tpath = DATA_ROOT / split / f"{split}_{source}.tsv"
    db = con()
    try:
        log(f"Blocking {split} {country} {source}")
        db.execute(f"CREATE TEMP TABLE q AS {normalized_record_sql(qpath, country, True, split == 'train')}")
        db.execute(f"CREATE TEMP TABLE t AS {normalized_record_sql(tpath, country, False, False)}")
        nq, nt = db.execute("SELECT (SELECT count(*) FROM q), (SELECT count(*) FROM t)").fetchone()
        log(f"  queries={nq:,}, targets={nt:,}")
        if not nq or not nt:
            db.execute(f"COPY (SELECT ''::VARCHAR qid, ''::VARCHAR tid, {', '.join(f'0::FLOAT {x}' for x in FEATURES)} WHERE false) TO {sq(out)} (FORMAT parquet, COMPRESSION zstd)")
            return out
        db.execute(f"CREATE TEMP TABLE tk AS {keys_sql('t')}")
        db.execute(f"CREATE TEMP TABLE qk AS {keys_sql('q')}")
        db.execute(f"""
            CREATE TEMP TABLE usable AS
            SELECT key, count(*) AS freq FROM tk GROUP BY key
            HAVING count(*) <= CASE WHEN starts_with(key, 'N=') OR starts_with(key, 'A=')
                THEN {MAX_EXACT_DF} ELSE {MAX_TOKEN_DF} END
        """)
        db.execute(f"""
            CREATE TEMP TABLE qku AS
            SELECT qk.rid, qk.key FROM qk JOIN usable USING (key)
            QUALIFY row_number() OVER (
              PARTITION BY qk.rid,
                CASE WHEN starts_with(qk.key, 'N=') THEN 'N='
                     WHEN starts_with(qk.key, 'A=') THEN 'A='
                     WHEN starts_with(qk.key, 'NT=') THEN 'NT='
                     WHEN starts_with(qk.key, 'AT=') THEN 'AT='
                     ELSE 'AN=' END
              ORDER BY usable.freq, qk.key
            ) <= CASE WHEN starts_with(qk.key, 'N=') OR starts_with(qk.key, 'A=')
                 THEN 1 ELSE {MAX_TOKEN_KEYS_PER_CHANNEL} END
        """)
        db.execute("""
            CREATE TEMP TABLE pairs AS
            SELECT qku.rid AS qid, tk.rid AS tid,
              count(*)::FLOAT AS key_hits,
              max(CASE WHEN starts_with(qku.key, 'N=') OR starts_with(qku.key, 'NT=') THEN 1 ELSE 0 END)::FLOAT AS name_key,
              max(CASE WHEN starts_with(qku.key, 'A=') OR starts_with(qku.key, 'AT=') THEN 1 ELSE 0 END)::FLOAT AS address_key,
              max(CASE WHEN starts_with(qku.key, 'AN=') THEN 1 ELSE 0 END)::FLOAT AS number_address_key
            FROM qku JOIN tk USING (key)
            GROUP BY qku.rid, tk.rid
        """)
        pair_count = db.execute("SELECT count(*) FROM pairs").fetchone()[0]
        log(f"  raw candidate pairs={pair_count:,}")
        feature_query = f"""
          WITH f AS (
            SELECT p.qid, p.tid, p.key_hits, p.name_key, p.address_key, p.number_address_key,
              jaro_winkler_similarity(q.nm, t.nm)::FLOAT AS name_jw,
              CASE WHEN q.ad = '' OR t.ad = '' THEN 0 ELSE jaro_winkler_similarity(q.ad, t.ad) END::FLOAT AS address_jw,
              CASE WHEN q.nm = '' OR t.nm = '' THEN 0
                ELSE jaccard(q.nm, t.nm) END::FLOAT AS name_jaccard,
              CASE WHEN q.ad = '' OR t.ad = '' THEN 0 ELSE jaccard(q.ad, t.ad) END::FLOAT AS address_jaccard,
              (least(length(q.nm), length(t.nm))::FLOAT / greatest(1, length(q.nm), length(t.nm)))::FLOAT AS name_length_ratio,
              (least(length(q.ad), length(t.ad))::FLOAT / greatest(1, length(q.ad), length(t.ad)))::FLOAT AS address_length_ratio,
              CASE WHEN q.num <> '' AND q.num = t.num THEN 1 ELSE 0 END::FLOAT AS number_equal,
              CASE WHEN q.num <> '' AND t.num <> '' THEN 1 ELSE 0 END::FLOAT AS number_both_present,
              CASE WHEN q.nm <> '' AND q.nm = t.nm THEN 1 ELSE 0 END::FLOAT AS exact_name,
              CASE WHEN q.ad <> '' AND q.ad = t.ad THEN 1 ELSE 0 END::FLOAT AS exact_address
            FROM pairs p JOIN q ON p.qid = q.rid JOIN t ON p.tid = t.rid
          )
          SELECT qid, tid, {', '.join(FEATURES)} FROM f
          QUALIFY row_number() OVER (
            PARTITION BY qid ORDER BY
              0.38 * greatest(name_jw, address_jw) + 0.28 * name_jw + 0.20 * address_jw + 0.07 * number_equal +
              0.05 * exact_name + 0.04 * least(key_hits, 3) / 3 DESC, tid
          ) <= {MAX_CANDIDATES_PER_SOURCE}
        """
        db.execute(f"COPY ({feature_query}) TO {sq(out)} (FORMAT parquet, COMPRESSION zstd)")
        kept = db.execute(f"SELECT count(*) FROM read_parquet({sq(out)})").fetchone()[0]
        log(f"  model-input pairs={kept:,} (cap {MAX_CANDIDATES_PER_SOURCE} per source/query)")
        return out
    finally:
        db.close()
        gc.collect()


def country_labels(split):
    db = con()
    try:
        path = DATA_ROOT / split / f"{split}_source1.tsv"
        return [r[0] for r in db.execute(f"SELECT DISTINCT country FROM {csv_source(path)} ORDER BY country").fetchall()]
    finally:
        db.close()


train_countries = country_labels("train")
test_countries = country_labels("test")
log(f"Country labels: train={train_countries}, test={test_countries}")
train_parts = []
test_parts = []
for country in train_countries:
    for source in ("source2", "source3"):
        train_parts.append(build_partition("train", country, source))
for country in test_countries:
    for source in ("source2", "source3"):
        test_parts.append(build_partition("test", country, source))


def parquet_files(parts):
    return "[" + ",".join(sq(p) for p in parts) + "]"


def make_labels():
    out = WORK / f"sampled_positive_links_mod{QUERY_SAMPLE_MOD}.parquet"
    if out.exists():
        return out
    db = con()
    try:
        gt = csv_source(DATA_ROOT / "train" / "train_ground_truth.tsv")
        db.execute(f"""
          COPY (
            SELECT source1_entity_id AS qid, unnest(str_split(matched_entity_ids, ',')) AS tid
            FROM {gt}
            WHERE hash(source1_entity_id) % 100 < {QUERY_SAMPLE_MOD}
              AND matched_entity_ids IS NOT NULL AND matched_entity_ids <> ''
          ) TO {sq(out)} (FORMAT parquet, COMPRESSION zstd)
        """)
        return out
    finally:
        db.close()


LABELS = make_labels()


def train_one(source, gpu):
    parts = [p for p in train_parts if p.name.startswith(source + "_")]
    db = con("5GB")
    try:
        files = parquet_files(parts)
        labels = sq(LABELS)
        cols = ", ".join("f." + x for x in FEATURES)
        common = f"""
          SELECT f.qid, f.tid, {cols}, CAST(l.tid IS NOT NULL AS UTINYINT) AS y
          FROM read_parquet({files}) f
          LEFT JOIN read_parquet({labels}) l ON f.qid = l.qid AND f.tid = l.tid
        """
        dev = db.execute(common + f" WHERE hash(f.qid) % 100 >= {VALID_MOD}").df()
        val = db.execute(common + f" WHERE hash(f.qid) % 100 < {VALID_MOD}").df()
        if dev.empty or val.empty or dev.y.nunique() < 2:
            raise RuntimeError(f"No usable training/validation rows for {source}")
        log(f"{source} GPU {gpu}: dev={len(dev):,}, val={len(val):,}, dev positives={int(dev.y.sum()):,}")
        train_x = np.ascontiguousarray(dev[FEATURES].to_numpy(dtype=np.float32))
        val_x = np.ascontiguousarray(val[FEATURES].to_numpy(dtype=np.float32))
        dtrain = xgb.DMatrix(train_x, label=dev.y.to_numpy(dtype=np.uint8), feature_names=FEATURES)
        dval = xgb.DMatrix(val_x, label=val.y.to_numpy(dtype=np.uint8), feature_names=FEATURES)
        del dev, train_x, val_x
        gc.collect()
        params = {
            "objective": "binary:logistic", "eval_metric": "aucpr", "tree_method": "hist",
            "device": f"cuda:{gpu}", "max_depth": 8, "eta": 0.07,
            "max_bin": 256, "subsample": 0.85, "colsample_bytree": 0.9,
            "min_child_weight": 8, "lambda": 4.0, "seed": SEED + gpu,
            "nthread": max(2, THREADS // 2),
        }
        booster = xgb.train(params, dtrain, num_boost_round=750,
                            evals=[(dval, "valid")], early_stopping_rounds=50,
                            verbose_eval=100)
        model_path = WORK / "models" / f"{source}.json"
        booster.save_model(str(model_path))
        score = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))
        valid_path = WORK / f"validation_{source}.parquet"
        pq.write_table(pa.table({
            "qid": val.qid.to_numpy(), "tid": val.tid.to_numpy(),
            "y": val.y.to_numpy(dtype=np.uint8), "score": score.astype(np.float32),
        }), valid_path, compression="zstd")
        log(f"{source}: best_iteration={booster.best_iteration}, validation scores saved")
        return str(model_path), str(valid_path)
    finally:
        db.close()
        gc.collect()


# Two independent, source-specific models fit concurrently on cuda:0 and cuda:1.
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(train_one, "source2", 0), pool.submit(train_one, "source3", 1)]
    trained = [f.result() for f in futures]
model_paths = {"source2": Path(trained[0][0]), "source3": Path(trained[1][0])}
validation_paths = [Path(trained[0][1]), Path(trained[1][1])]


validation_db = con()
gt = csv_source(DATA_ROOT / "train" / "train_ground_truth.tsv")
validation_db.execute(f"""
  CREATE TEMP TABLE truth AS
  SELECT source1_entity_id AS qid,
    CASE WHEN matched_entity_ids IS NULL OR matched_entity_ids = ''
      THEN 0 ELSE length(str_split(matched_entity_ids, ',')) END AS true_n
  FROM {gt} WHERE hash(source1_entity_id) % 100 < {VALID_MOD}
""")
validation_db.execute(f"CREATE TEMP VIEW scores AS SELECT * FROM read_parquet({parquet_files(validation_paths)})")
total_true = validation_db.execute("SELECT sum(true_n) FROM truth").fetchone()[0]
recalled = validation_db.execute("SELECT sum(y) FROM scores").fetchone()[0] or 0
candidate_recall = float(recalled / total_true)


def validation_metric(threshold):
    out = validation_db.execute(f"""
      WITH predicted AS (
        SELECT qid, count(*) AS pred_n, sum(y) AS hit_n
        FROM scores WHERE score >= {float(threshold)} GROUP BY qid
      )
      SELECT avg(CASE WHEN true_n = 0 AND coalesce(pred_n,0) = 0 THEN 1.0
        ELSE 1.25 * coalesce(hit_n,0) / (0.25 * true_n + coalesce(pred_n,0)) END),
        sum(CASE WHEN true_n=0 AND coalesce(pred_n,0)=0 THEN 1 ELSE 0 END),
        sum(CASE WHEN true_n=0 THEN 1 ELSE 0 END)
      FROM truth LEFT JOIN predicted USING(qid)
    """).fetchone()
    return {"threshold": float(threshold), "macro_f05": float(out[0]),
            "correct_singletons": int(out[1]), "singletons": int(out[2]),
            "candidate_link_recall": candidate_recall}


grid = np.r_[np.arange(0.05, 0.951, 0.05), [0.975, 0.99, 0.995]]
first = [validation_metric(t) for t in grid]
initial_best = max(first, key=lambda r: (r["macro_f05"], r["threshold"]))
fine = np.arange(max(0.005, initial_best["threshold"] - 0.045),
                 min(0.999, initial_best["threshold"] + 0.046), 0.005)
metrics = first + [validation_metric(t) for t in fine]
best = max(metrics, key=lambda r: (r["macro_f05"], r["threshold"]))
threshold = best["threshold"]
validation_db.close()
ao_atomic_json(WORK / "validation_summary.json", best)
log(f"Validation: {best}")
if best["candidate_link_recall"] < 0.98:
    log("WARNING: blocking recall below 98%; inspect missed links before a portal upload.")


def predict_one(source, gpu):
    booster = xgb.Booster()
    booster.load_model(str(model_paths[source]))
    booster.set_param({"device": f"cuda:{gpu}", "nthread": max(2, THREADS // 2)})
    best_iteration = booster.attr("best_iteration")
    prediction_rounds = int(best_iteration) + 1 if best_iteration is not None else booster.num_boosted_rounds()
    model_hash = ao_sha256(model_paths[source])
    source_parts = [p for p in test_parts if p.name.startswith(source + "_")]
    accepted_files = []
    for part in source_parts:
        out = WORK / "accepted" / part.name
        partial = out.with_name(out.stem + ".partial.parquet")
        marker = out.with_name(out.name + ".complete.json")
        # Never present a partially scored partition as a completed checkpoint.
        marker.unlink(missing_ok=True)
        candidate_fingerprint = ao_fingerprint(part)
        candidate_hash = ao_sha256(part)
        accepted_count = scored_count = 0
        with pq.ParquetFile(part) as reader, pq.ParquetWriter(
            partial, pa.schema([pa.field("qid", pa.string()), pa.field("tid", pa.string())]),
            compression="zstd",
        ) as writer:
            expected_rows = reader.metadata.num_rows
            for batch in reader.iter_batches(batch_size=200_000, columns=["qid", "tid"] + FEATURES):
                frame = batch.to_pandas()
                matrix = xgb.DMatrix(np.ascontiguousarray(frame[FEATURES].to_numpy(dtype=np.float32)), feature_names=FEATURES)
                score = booster.predict(matrix, iteration_range=(0, prediction_rounds))
                scored_count += len(frame)
                keep = score >= threshold
                if keep.any():
                    selected = frame.loc[keep, ["qid", "tid"]]
                    writer.write_table(pa.Table.from_pandas(selected, preserve_index=False))
                    accepted_count += len(selected)
                    del selected
                del frame, matrix, score
        if scored_count != expected_rows or candidate_fingerprint != ao_fingerprint(part):
            raise RuntimeError(f"Incomplete inference or changed candidates: {part}")
        if ao_sha256(model_paths[source]) != model_hash:
            raise RuntimeError(f"Model changed during inference: {source}")
        os.replace(partial, out)
        ao_atomic_json(marker, {
            "candidate": candidate_fingerprint, "candidate_sha256": candidate_hash,
            "scored_rows": scored_count,
            "accepted_rows": accepted_count, "accepted_sha256": ao_sha256(out),
            "model_file": model_paths[source].name, "model_sha256": model_hash,
            "threshold": threshold, "prediction_rounds": prediction_rounds,
        })
        log(f"Inference {source} {part.name}: accepted {accepted_count:,}; checkpoint complete")
        accepted_files.append(out)
    return accepted_files


with ThreadPoolExecutor(max_workers=2) as pool:
    f2 = pool.submit(predict_one, "source2", 0)
    f3 = pool.submit(predict_one, "source3", 1)
    accepted_parts = f2.result() + f3.result()


gc.collect()
torch.cuda.empty_cache()
assembly_result = assemble_outputs(
    test_parts, accepted_parts, DATA_ROOT, WORK,
    max_candidates_per_entity=2 * MAX_CANDIDATES_PER_SOURCE,
)

manifest = {
    "data_root": str(DATA_ROOT), "country_labels_train": train_countries,
    "country_labels_test": test_countries, "validation": best,
    "model_license": "Apache-2.0 (XGBoost)", "xgboost": xgb.__version__,
    "duckdb": duckdb.__version__, "pandas": pd.__version__,
    "pyarrow": pa.__version__, "torch": torch.__version__,
    "gpu_names": [torch.cuda.get_device_name(i) for i in (0, 1)],
    "query_sample_mod": QUERY_SAMPLE_MOD, "valid_mod": VALID_MOD,
    "max_token_df": MAX_TOKEN_DF, "max_exact_df": MAX_EXACT_DF,
    "max_token_keys_per_channel": MAX_TOKEN_KEYS_PER_CHANNEL,
    "max_candidates_per_source": MAX_CANDIDATES_PER_SOURCE,
    "output": assembly_result["outputs"],
    "output_validation": assembly_result["validation"],
    "official_validator_run": assembly_result["official_validator_run"],
    "assembly_version": assembly_result["assembly_version"],
}
ao_atomic_json(WORK / "run_manifest.json", manifest)
log(f"Finished. Upload only {OUTPUT / 'matching_results.tsv'} to the portal after review.")
log(f"Final package also needs {OUTPUT / 'candidate_pairs.tsv'} and runnable code.")
